# MONTERREY MOVE — real-time classifier (Google Colab)

Replays the **training** sessions packet by packet (20 Hz, as the game server sends them) through the frozen `LogReg_todas` model. Uses only `dat1/raw`; the sealed test data lives in Azure and is never opened here.

In [ ]:
!git clone -q https://github.com/jojogarciaortega2707/MonterreyMove_sEMG.git repo
%cd repo
!pip -q install scikit-learn==1.9.1

## 1. Parity checks
Envelope vs the app's CSV, features vs `code/03_clasificador.py`, JSON model vs frozen pkl.

In [ ]:
!python realtime/pruebas.py

## 2. Real-time path, amplitude labels (training only — circular, see README)

In [ ]:
!python realtime/rt_evaluar.py dat1/raw /content/rt_amplitud --amplitud dat1/monterrey_move_ML_unido.csv

## 3. Real-time path, calibration-script labels (dry run)
Script start estimated by `code/05_guion_calibracion.py` (`results/guion_calibracion.csv`). S01 and S07 are ambiguous.

In [ ]:
import pandas as pd
g = pd.read_csv('results/guion_calibracion.csv')
g[['id_sesion']].assign(inicio_guion_s=g.t0_mejor).to_csv('/content/guion_ensayo.csv', index=False)
!python realtime/rt_evaluar.py dat1/raw /content/rt_guion --guion /content/guion_ensayo.csv

## 4. One session, live view

In [ ]:
import sys, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, 'realtime')
import envolvente as E, rt_modelo as M
sid = 'dat1/raw/MM-0001_20261110-100929'
t, uv = E.paquetes(E.envolvente(np.load(sid + '_senal_cruda.npy').astype(float)))
i0 = 170 * 20                      # files are cumulative: this session starts at second 170
t, uv = t[i0:] - t[i0], uv[i0:]
cl = M.ClasificadorEnVivo()
p = np.array([np.nan if (r := cl.agregar(v)) is None else r['prob_contraccion'] for v in uv])
fig, ax = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
ax[0].plot(t, uv, lw=.8, color='#52514e'); ax[0].set_ylabel('envelope (µV)')
ax[1].fill_between(t, 0, p, step='post', color='#eb6834', alpha=.7); ax[1].axhline(.5, ls='--', c='k', lw=.6)
ax[1].set_ylabel('P(contraction)'); ax[1].set_xlabel('s'); plt.tight_layout()